In [ ]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from inference import ConditionSpec, predict_scale_up
from models import DDPM


In [ ]:
class ZeroSliceUNet(nn.Module):
    def forward(self, z_t, t, condition_z, axis, slice_index):
        return torch.zeros_like(z_t)


class UpsampleVAE(nn.Module):
    def encode(self, x):
        mu = F.avg_pool2d(x, kernel_size=4, stride=4).repeat(1, 4, 1, 1)
        return mu, torch.zeros_like(mu)

    def decode(self, z):
        return F.interpolate(z[:, :1], scale_factor=4, mode="nearest").clamp(0, 1)


In [ ]:
condition = torch.ones(1, 1, 128, 128)
result = predict_scale_up(
    vae=UpsampleVAE(),
    unet=ZeroSliceUNet(),
    ddpm=DDPM(timesteps=1),
    conditions=[ConditionSpec(condition=condition, axis=0, slice_index=32)],
    output_size=128,
    latent_ch=4,
)

print(result["volume_z"].shape, result["volume"].shape, result["condition_errors"])
assert result["volume_z"].shape == torch.Size([4, 32, 32, 32])
assert result["volume"].shape == torch.Size([128, 1, 128, 128])
assert result["condition_errors"] == [0.0]
